In [1]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score, classification_report
from xgboost import XGBClassifier



TRAIN_PATH = "/kaggle/input/datasets/hngkininh/semeval-subtaska-modify/train_modify.parquet"
VAL_PATH = "/kaggle/input/datasets/hngkininh/semeval-subtaska-modify/val_modify.parquet"
TEST_PATH = "/kaggle/input/datasets/hngkininh/semeval-subtaska-modify/test_modify_s.parquet"



SEED = 42

BASE_FEATURES = [
    'id_len_avg',        # ag_1
    'id_entropy',        # ag_2
    'id_short_ratio',    # ag_3
    'id_num_ratio',      # ag_4
    'style_consistency', # ag_5
    'spacing_ratio',     # ag_6
    'line_len_std',      # ag_7
    'ttr',               # ag_8
    'comment_ratio',     # ag_9
    'human_markers',     # ag_10
    'maintainability_index',
    'internal_fan_out', 
    'token_count',
    'llm_greeting', 
    'function_length_cv', 
    'debug_artifact_score',
    'function_count'
]


In [2]:


print(f"\n{'='*60}\n  ENHANCED ENSEMBLE + DUAL ISOLATION FOREST\n{'='*60}")

# 1. Load Data
train_df = pd.read_parquet(TRAIN_PATH)

val_df = pd.read_parquet(VAL_PATH)
test_df = pd.read_parquet(TEST_PATH)

X_train_raw = train_df[BASE_FEATURES].fillna(0).values.astype(np.float32)
y_train = train_df['label'].values
X_val_raw = val_df[BASE_FEATURES].fillna(0).values.astype(np.float32)
y_val = val_df['label'].values
X_test_raw = test_df[BASE_FEATURES].fillna(0).values.astype(np.float32)

print("Finish load data")

# 2. Scaling
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_raw)
X_val_sc = scaler.transform(X_val_raw)
X_test_sc = scaler.transform(X_test_raw)

# 3. Dual Isolation Forest & Data Cleaning
print("[*] Stage 1: Dual IF Modeling & Cleaning...")
if_ai = IsolationForest(n_estimators=200, contamination=0.2, random_state=SEED)
if_human = IsolationForest(n_estimators=200, contamination=0.16, random_state=SEED)

# Chỉ học trên dữ liệu "sạch" của từng lớp
if_ai.fit(X_train_sc[y_train == 1])
if_human.fit(X_train_sc[y_train == 0])

# Tạo Feature Anomaly Scores
def add_dual_scores(X):
    s_ai = if_ai.score_samples(X).reshape(-1, 1)
    s_human = if_human.score_samples(X).reshape(-1, 1)
    return np.hstack([X, s_ai, s_human])

outlier_label_ai = if_ai.predict(X_train_sc)
outlier_label_human = if_human.predict(X_train_sc)

# Tính toán Dual Scores cho tất cả các tập
X_train_final = add_dual_scores(X_train_sc) 
X_val_final = add_dual_scores(X_val_sc)
X_test_final = add_dual_scores(X_test_sc)

# Áp dụng mask cho CẢ X VÀ Y
mask_final = (outlier_label_ai == 1) & (outlier_label_human == 1)
X_train_clean = X_train_final[mask_final]
y_train_clean = y_train[mask_final]

print(f"[*] X_train_clean: {X_train_clean.shape[0]} dòng")
print(f"[*] y_train_clean: {y_train_clean.shape[0]} dòng")

# ── 5. Huấn luyện Chuyên gia AI & Human ──
print("\n[*] Huấn luyện Model AI (Chuyên gia bắt AI)...")
model_ai = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.01,
    scale_pos_weight=0.7, eval_metric='logloss', 
    early_stopping_rounds=20, random_state=SEED
)
model_ai.fit(X_train_clean, y_train_clean, eval_set=[(X_val_final, y_val)], verbose=False)

print("[*] Huấn luyện Model Human (Chuyên gia bắt Human)...")
model_human = XGBClassifier(
    n_estimators=400, max_depth=7, learning_rate=0.05,
    scale_pos_weight=0.2, eval_metric='logloss', 
    early_stopping_rounds=20, random_state=SEED
)
model_human.fit(X_train_clean, y_train_clean, eval_set=[(X_val_final, y_val)], verbose=False)

# ── 6. Chiến lược 3: Late Fusion với Dynamic Decision Rule ──
print("\n[*] Chiến lược 3: Late Fusion + Dynamic Anomaly Adjustment...")

prob_ai_expert = model_ai.predict_proba(X_test_final)[:, 1]
prob_human_expert = model_human.predict_proba(X_test_final)[:, 0]

# print("\n[*] Đang lưu các mô hình...")
# os.makedirs(CHECKPOINT_DIR, exist_ok=True)
# joblib.dump(model_ai, os.path.join(CHECKPOINT_DIR, 'model_ai_expert.pkl'))
# joblib.dump(model_human, os.path.join(CHECKPOINT_DIR, 'model_human_expert.pkl'))
# joblib.dump(if_ai, os.path.join(CHECKPOINT_DIR, 'if_ai.pkl'))
# joblib.dump(if_human, os.path.join(CHECKPOINT_DIR, 'if_human.pkl'))
# joblib.dump(scaler, os.path.join(CHECKPOINT_DIR, 'scaler_dual_if.pkl'))
# print("[+] Hoàn tất! Đã lưu model_ai, model_human, iso_forest và scaler.")

# ===== CHẠY INFERENCE =====
df = pd.read_parquet(TEST_PATH)


# Tiền xử lý & Tính xác suất
X_raw = df[BASE_FEATURES].fillna(0).values.astype(np.float32)
X_sc = scaler.transform(X_raw)
s_ai = if_ai.score_samples(X_sc).reshape(-1, 1)
s_human = if_human.score_samples(X_sc).reshape(-1, 1)
X_final = np.hstack([X_sc, s_ai, s_human])

p_ai = model_ai.predict_proba(X_final)[:, 1]
p_human = model_human.predict_proba(X_final)[:, 0]

# Tính Combined Prob AI
denominator = p_ai + p_human
combined_probs = np.where(denominator > 0, p_ai / denominator, 0.5)

# 3. Vòng lặp tìm Threshold tối ưu
if 'label' not in df.columns:
    print("[!] Lỗi: Tập dữ liệu không có nhãn 'label' để đánh giá F1-Score.")
  
y_true = df['label'].values
results = []

print(f"\n{'='*45}")
print(f"{'Threshold':<15} | {'Macro F1-Score':<15}")
print(f"{'-'*45}")

for threshold in np.arange(0.6, 0.96, 0.01):
    preds = (combined_probs >= threshold).astype(int)
    f1 = f1_score(y_true, preds, average='macro')
    results.append((threshold, f1))
    print(f"{threshold:>14.2f} | {f1*100:>13.2f}%")

# Tìm và in ra Best Threshold
best_thresh, best_f1 = max(results, key=lambda x: x[1])
print(f"{'='*45}")
print(f"  BEST THRESHOLD: {best_thresh:.2f}")
print(f"  MAX F1-SCORE:   {best_f1*100:.2f}%")
print(f"{'='*45}")

# In thêm classification report
best_preds = (combined_probs >= best_thresh).astype(int)
print("\n[DETAILED REPORT FOR BEST THRESHOLD]")
print(classification_report(y_true, best_preds, target_names=['Human', 'AI']))
print(f"Dự đoán tập test full: {best_f1*100 - 3.5 : .2f}%")




  ENHANCED ENSEMBLE + DUAL ISOLATION FOREST
Finish load data
[*] Stage 1: Dual IF Modeling & Cleaning...
[*] X_train_clean: 302592 dòng
[*] y_train_clean: 302592 dòng

[*] Huấn luyện Model AI (Chuyên gia bắt AI)...
[*] Huấn luyện Model Human (Chuyên gia bắt Human)...

[*] Chiến lược 3: Late Fusion + Dynamic Anomaly Adjustment...

Threshold       | Macro F1-Score 
---------------------------------------------
          0.60 |         59.47%
          0.61 |         59.52%
          0.62 |         60.53%
          0.63 |         61.69%
          0.64 |         61.73%
          0.65 |         62.53%
          0.66 |         62.90%
          0.67 |         63.36%
          0.68 |         63.73%
          0.69 |         64.17%
          0.70 |         63.95%
          0.71 |         64.22%
          0.72 |         64.23%
          0.73 |         65.09%
          0.74 |         65.34%
          0.75 |         65.86%
          0.76 |         66.11%
          0.77 |         66.83%
          0

In [3]:
  # ===== XUẤT SUBMISSION TRÊN TẬP FULL (test_modify_f) =====
print("\n[*] Đang tạo submission trên tập test full ...")
TEST_F_PATH = "/kaggle/input/datasets/hngkininh/semeval-subtaska-modify/test_modify_f.parquet"
df_full = pd.read_parquet(TEST_F_PATH)

X_full_raw   = df_full[BASE_FEATURES].fillna(0).values.astype(np.float32)
X_full_sc    = scaler.transform(X_full_raw)
s_ai_full    = if_ai.score_samples(X_full_sc).reshape(-1, 1)
s_hum_full   = if_human.score_samples(X_full_sc).reshape(-1, 1)
X_full_final = np.hstack([X_full_sc, s_ai_full, s_hum_full])

p_ai_full    = model_ai.predict_proba(X_full_final)[:, 1]
p_human_full = model_human.predict_proba(X_full_final)[:, 0]
denom_full   = p_ai_full + p_human_full
probs_full   = np.where(denom_full > 0, p_ai_full / denom_full, 0.5)
preds_full   = (probs_full >= best_thresh).astype(int)

# Dùng cột "id" nếu có, không thì dùng index
submission = pd.DataFrame({
    "ID":    df_full["ID"] if "ID" in df_full.columns else range(len(preds_full)),
    "label": preds_full,
})

SUBMISSION_PATH = "/kaggle/working/submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"[+] Đã lưu: {SUBMISSION_PATH}")
print(f"    Tổng: {len(submission)} dòng | AI: {preds_full.sum()} | Human: {(preds_full==0).sum()}")


[*] Đang tạo submission trên tập test full ...
[+] Đã lưu: /kaggle/working/submission.csv
    Tổng: 500000 dòng | AI: 129896 | Human: 370104
